# Semana 18 — LoRA do Zero: a Matemática do Low Rank
**Fase 4: Parameter-Efficient Fine-Tuning (PEFT)** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/04-Phase-4-Fine-Tuning/notebooks/semana_18_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-18-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/04-Phase-4-Fine-Tuning/docs/Semana-18-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_18_gabarito.ipynb`.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, math, copy, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
torch.manual_seed(0)

## Passo 1 — Low rank na prática
Construa uma matriz 64×64 de rank 4 (mais um pouco de ruído), calcule a SVD e veja os singular values.

In [ ]:
U, V = torch.randn(64, 4), torch.randn(4, 64)
M = U @ V + 0.01 * torch.randn(64, 64)
# [SEU CÓDIGO AQUI] singular values de M
plt.semilogy(S, "o-"); plt.xlabel("índice"); plt.ylabel("valor singular"); plt.show()
print("posto numérico:", int((S > 0.5).sum()), "| parâmetros cheios:", 64 * 64, "| fatorados (r=4):", 4 * (64 + 64))

## Passo 2 — A LoRA layer
`LoRALinear` embrulha uma `nn.Linear` existente: congela os pesos originais e soma `(alpha / r) · B(A x)`.
- `A`: shape `(r, in_features)`, inicialização `kaiming_uniform_` com `a=math.sqrt(5)`.
- `B`: shape `(out_features, r)`, **zeros**.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.escala = alpha / r
        # [SEU CÓDIGO AQUI] crie self.A e self.B como nn.Parameter
    def forward(self, x):
        # [SEU CÓDIGO AQUI] saída original + low-rank update
    def delta(self):
        return self.escala * self.B @ self.A

lin = nn.Linear(32, 16); lora = LoRALinear(lin, r=4); x = torch.randn(5, 32)
assert torch.allclose(lora(x), lin(x)), "no início, LoRA não deve alterar a saída"
print("treináveis:", sum(p.numel() for p in lora.parameters() if p.requires_grad), "| congelados:", sum(p.numel() for p in lin.parameters()))

## Passo 3 — Um modelo "pré-treinado"
Treinamos um MLP nos dígitos manuscritos 8×8. Ele será nosso base model.

In [ ]:
X, y = load_digits(return_X_y=True); X = torch.tensor(X / 16.0, dtype=torch.float32); y = torch.tensor(y)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

def criar(): return nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
def treinar(modelo, X, y, passos=400, lr=1e-3):
    opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    for _ in range(passos):
        idx = torch.randint(len(X), (64,)); loss = F.cross_entropy(modelo(X[idx]), y[idx]); opt.zero_grad(); loss.backward(); opt.step()
    return modelo
@torch.no_grad()
def acc(modelo, X, y): return (modelo(X).argmax(1) == y).float().mean().item()

base = treinar(criar(), Xtr, ytr); print("acurácia do modelo base na tarefa original:", round(acc(base, Xte, yte), 3))

## Passo 4 — Uma tarefa nova: imagens rotacionadas
O "domínio corporativo" aqui são os mesmos dígitos girados 90°. O base model não foi treinado para isso.

In [ ]:
girar = lambda X: torch.rot90(X.view(-1, 8, 8), 1, dims=(1, 2)).reshape(-1, 64)
Xtr_n, Xte_n = girar(Xtr), girar(Xte)
print("modelo base na tarefa nova (sem ajuste):", round(acc(base, Xte_n, yte), 3))

## Passo 5 — Fine-tuning completo vs. LoRA
Escreva `aplicar_lora`: copia o modelo e troca cada `nn.Linear` por `LoRALinear`. Compare trainable parameters e accuracy.

In [ ]:
def aplicar_lora(modelo, r=4, alpha=8):
    m = copy.deepcopy(modelo)
    # [SEU CÓDIGO AQUI] substitua cada nn.Linear por LoRALinear
    return m

def treinaveis(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

torch.manual_seed(0)
completo = treinar(copy.deepcopy(base), Xtr_n, ytr)
com_lora = treinar(aplicar_lora(base, r=4), Xtr_n, ytr, lr=3e-3)
for nome, m in [("fine-tuning completo", completo), ("LoRA r=4", com_lora)]:
    print(f"{nome:22s} | treináveis: {treinaveis(m):7,} ({treinaveis(m) / treinaveis(completo):6.1%}) | tarefa nova: {acc(m, Xte_n, yte):.3f} | tarefa original: {acc(m, Xte, yte):.3f}")

Observe a última coluna: o que aconteceu com a tarefa original em cada caso? Com LoRA, basta desligar o adapter para ter o base model de volta, intacto.

## Passo 6 — O efeito do rank

In [ ]:
resultados = []
for r in (1, 2, 4, 8, 16, 32):
    torch.manual_seed(0)
    # [SEU CÓDIGO AQUI] aplique LoRA com rank r (alpha = 2r) e treine na tarefa nova
    resultados.append((r, treinaveis(m), acc(m, Xte_n, yte))); print(resultados[-1])
plt.plot([r[0] for r in resultados], [r[2] for r in resultados], "o-"); plt.axhline(acc(completo, Xte_n, yte), ls="--", c="gray", label="fine-tuning completo")
plt.xscale("log", base=2); plt.xlabel("posto r"); plt.ylabel("acurácia na tarefa nova"); plt.legend(); plt.show()

## Passo 7 — Merge: sem custo na inference
Escreva `fundir`: para cada `LoRALinear`, some `delta()` ao peso da layer base e devolva uma `nn.Linear` comum.

In [ ]:
@torch.no_grad()
def fundir(modelo):
    m = copy.deepcopy(modelo)
    for i, camada in enumerate(m):
        if isinstance(camada, LoRALinear):
            # [SEU CÓDIGO AQUI] some delta() ao peso original e substitua a layer pela nn.Linear base
    return m

fundido = fundir(com_lora)
assert torch.allclose(fundido(Xte_n), com_lora(Xte_n), atol=1e-5)
print("saídas idênticas | camadas do modelo fundido:", [type(c).__name__ for c in fundido])

## Passo 8 — A atualização aprendida tem mesmo low rank?
Compare os singular values de `ΔW` do full fine-tuning (primeira layer) com os do LoRA.

In [ ]:
dW_completo = completo[0].weight.detach() - base[0].weight.detach()
dW_lora = com_lora[0].delta().detach()
# [SEU CÓDIGO AQUI] singular values das duas atualizações
plt.semilogy(s_c / s_c[0], label="ΔW fine-tuning completo"); plt.semilogy(s_l[:8] / s_l[0], "o-", label="ΔW LoRA (r=4)")
plt.xlabel("índice"); plt.ylabel("valor singular (normalizado)"); plt.legend(); plt.show()
energia = (s_c[:8] ** 2).sum() / (s_c ** 2).sum(); print(f"os 8 maiores valores singulares concentram {energia:.0%} da energia de ΔW completo")

## Passo 9 — Conta para um LLM de verdade
Um modelo com `d = 4096` e 32 layers, LoRA em `q_proj` e `v_proj` (ambas `4096 × 4096`). Quantos trainable parameters com `r = 16`? Quanto ocupa o adapter em disco em `float16`?

In [ ]:
# [SEU CÓDIGO AQUI] calcule e imprima